## 1. Azure で Entity Graph を検索する

Entity Graph では、オントロジーで定めた型と述語を使って、原文から抽出された実体と関係をたどります。この Notebook では「どの施設がどの指定や登録を受けていて、どこにあるか」を、手書きの GQL（Graph Query Language）と自然言語による Agentic Retrieval の 2 通りで調べ、結果を根拠チャンクまでたどります。

```mermaid
flowchart TD
    D[手書きの GQL] --> G[管理された Entity Graph を探索]
    Q[自然言語の質問] --> KB[ナレッジ ベースの retrieve]
    KB --> P[モデルが質問を GQL に変換]
    P --> G
    G --> R[グラフ行と根拠チャンク]
    P --> A[activity で生成 GQL と実行履歴を確認]
```

| 確認対象 | 直接 GQL（3 章） | Agentic Retrieval（4 章） |
|---|---|---|
| 入力 | 型と述語を指定した手書きの GQL | `messages` に入れた自然言語の質問 |
| 探索先 | ナレッジ ソースが生成したグラフ インデックス | グラフ抽出を有効にした `azureBlob` ソースを参照する KB |
| 観察する出力 | グラフ行、診断、各行からたどれる根拠チャンク | `activity` の生成 GQL、`references` のグラフ行と根拠チャンク |

直接 GQL で経路が見つかるのに自然言語検索で見つからないときは、`activity` の生成 GQL と手書きの GQL を比較すると、型・述語・絞り込み条件のどこが異なるかを確認できます。


この Notebook は機能テスト 1 と同じ **`azure-search-documents==12.2.0b1`** を使います。`SearchIndexClient` で定義を読み取り、`SearchClient.graph()` で直接 GQL、`KnowledgeBaseRetrievalClient.retrieve()` で Agentic Retrieval を実行します。要求と応答は SDK モデルの `.as_dict()` で確認できます。Copilot プラグインは不要です。

出典: [Entity Graph](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph) / [検索の推論レベル](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)

## 2. 接続設定と変更の境界

### Azure 側の準備

| 項目 | 必要なもの |
|---|---|
| サービス | `2026-11-01-preview` とグラフ機能に対応する Foundry IQ / Azure AI Search |
| Entity Graph | 機能テスト 1 で同期済みのグラフ対応ソース。既定名は `ft-entity-ks` です。別のソース名なら `FOUNDRY_IQ_ENTITY_SOURCE` を設定し、別のオントロジーなら 3 章の GQL と 4 章の質問をその定義に合わせます |
| 検索モデル（4 章のみ） | Azure OpenAI のチャット デプロイ。グラフ抽出時に使ったモデルとは別に、ナレッジ ベース側で指定します |
| 実行ユーザー | Microsoft Entra ID で認証します。検索には Search Index Data Reader、ソースやインデックス定義の参照・KB 作成には Search Service Contributor の対応する権限を確認します |
| Search のマネージド ID（4 章のみ） | モデルに接続するため、Azure OpenAI に対する Cognitive Services OpenAI User の権限が必要です |

この Notebook はサービスやモデルの作成、ロール付与、文書のアップロード、グラフの再抽出、リソース削除を行いません。グラフの準備は [機能テスト 1](./functional-test-1-entity-graph.ipynb) を参照してください。

### カーネルの依存パッケージ

ライブ用セルで import エラーが出る場合、または SDK が `12.2.0b1` でない場合は、選択した Notebook カーネルで次を実行してください。

```python
%pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv
```

[.env.example](./.env.example) を参考に、Notebook と同じフォルダーの `.env` に必要な項目を設定します。3 章には `AZURE_SEARCH_ENDPOINT` が必要です。4 章では `AZURE_OPENAI_ENDPOINT`、`AZURE_OPENAI_DEPLOYMENT`、`AZURE_OPENAI_MODEL` も設定します。`DefaultAzureCredential` が使う対象テナントの認証を準備し、資格情報やアクセストークンをセルに書かないでください。`.env` に指定した設定を環境変数より優先します。

### 実行スイッチ

- `RUN_AZURE`（この章）: 既定の `True` で Azure に接続し、既存グラフを読み取って検索します。Azure に接続しない場合は `False` にします。Azure 上の検索と 4 章のモデル呼び出しには料金が発生する場合があります。
- `CONFIRM_WRITES`（4.1 の作成セル）: 計画を確認した後に、この Notebook 用の KB の新規作成だけを許可します。既存ソースや KB は上書きしません。

新しいカーネルで始める場合は、この章のコードセルを上から実行してください。4 章の自然言語検索だけを試す場合は、3 章の直接 GQL を飛ばせます。

出典: [Entity Graph の前提と制限](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [ナレッジ ベースの検索権限](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [Search の RBAC](https://learn.microsoft.com/azure/search/search-security-rbac) / [ローカルでの Azure Identity 認証](https://learn.microsoft.com/azure/developer/python/sdk/authentication/local-development-environment)

In [ ]:
from __future__ import annotations

import html
import json
import os
import re
from pathlib import Path
from urllib.parse import urlsplit

from IPython.display import HTML, SVG, display


NOTEBOOK_DIR = Path(".")


def show_json(value: object) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def preview_text(value: object) -> object:
    return value[:200] + "…" if isinstance(value, str) and len(value) > 200 else value


def table(rows: list[dict]) -> None:
    if not rows:
        print("表示対象は 0 件です。")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = "".join(f"<th>{html.escape(str(key))}</th>" for key in columns)
    body = []
    for row in rows:
        cells = []
        for key in columns:
            value = row.get(key, "")
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
            cells.append(f"<td>{html.escape(text)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

In [ ]:
RUN_AZURE = True
API_VERSION = "2026-11-01-preview"

if RUN_AZURE:
    from importlib.metadata import version
    from azure.core import MatchConditions
    from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
    from azure.core.pipeline import PipelineResponse
    from azure.identity import DefaultAzureCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.indexes.models import (
        AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
        KnowledgeSourceReference,
    )
    from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
    from azure.search.documents.knowledgebases.models import (
        AgentRuntimeGraphHydrateDocumentsParams, AgentRuntimeGraphParams,
        AgentRuntimeGraphProvenanceParams, AzureBlobKnowledgeSourceParams,
        KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
        KnowledgeRetrievalLowReasoningEffort,
    )
    from azure.search.documents.models import (
        GraphQueryProvenanceHydrateDocumentsParameters, GraphQueryProvenanceParameters,
    )
    from dotenv import load_dotenv
    if version("azure-search-documents") != "12.2.0b1":
        raise RuntimeError("機能テスト 1 と同じ azure-search-documents==12.2.0b1 をカーネルに導入してください。")
    load_dotenv(NOTEBOOK_DIR / ".env", override=True)

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
MODEL_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "")
CHAT_MODEL = os.getenv("AZURE_OPENAI_MODEL", "")
ENTITY_SOURCE = os.getenv("FOUNDRY_IQ_ENTITY_SOURCE", "ft-entity-ks")
ENTITY_KB = os.getenv("FOUNDRY_IQ_ENTITY_KB", "ft-entity-agentic-kb")

if RUN_AZURE:
    endpoint = urlsplit(SEARCH_ENDPOINT)
    if (endpoint.scheme != "https" or not endpoint.hostname or endpoint.username
            or endpoint.password or endpoint.path or endpoint.query or endpoint.fragment or "<" in SEARCH_ENDPOINT):
        raise ValueError("AZURE_SEARCH_ENDPOINT に実際の HTTPS サービス URL を設定してください。")
    credential = DefaultAzureCredential()
    client_options = {"api_version": API_VERSION, "retry_total": 0, "connection_timeout": 10, "read_timeout": 90}
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, **client_options)
    azure_clients: list[SearchIndexClient | SearchClient | KnowledgeBaseRetrievalClient] = [index_client]

table([{"Azure 実行": RUN_AZURE, "API バージョン": API_VERSION,
        "Entity ソース": ENTITY_SOURCE, "Entity Graph 用 KB": ENTITY_KB}])

### 2.1 SDK の例外処理と安全な新規作成

認証と通信は SDK に任せます。サービスのエラーは SDK の `HttpResponseError` で停止し、存在しないリソースだけ `ResourceNotFoundError` として扱います。`retrieve` の部分応答 `206` は応答フックで検出して停止します。各クライアントの `retry_total = 0` により、自動再試行は行いません。

KB の新規作成には `MatchConditions.IfMissing` を使い、同名の KB が先に作られていても上書きしません。すでに存在する場合は、この Notebook の計画に含まれる設定値を照合してから再利用します。サーバーが追加する既定値は照合対象外です。

出典: [Search の同時実行制御](https://learn.microsoft.com/azure/search/search-howto-concurrency) / [Knowledge Retrieval REST](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview)

In [ ]:
def reject_partial_response(response: PipelineResponse) -> None:
    if response.http_response.status_code == 206:
        raise HttpResponseError("Agentic Retrieval が部分応答を返しました。応答本文と実行履歴を確認してください。",
                                response=response.http_response)


def require_expected(actual: object, expected: object, path: str = "resource") -> None:
    if isinstance(expected, dict) and isinstance(actual, dict):
        for key, value in expected.items():
            require_expected(actual.get(key), value, f"{path}.{key}")
    elif isinstance(expected, list) and isinstance(actual, list) and len(actual) == len(expected):
        for index, value in enumerate(expected):
            require_expected(actual[index], value, f"{path}[{index}]")
    elif actual != expected:
        raise ValueError(f"既存設定が計画と異なります: {path}。上書きせず、別名を使うか設定を確認してください。")


def create_or_reuse_kb(definition: KnowledgeBase, *, confirm_writes: bool) -> KnowledgeBase:
    try:
        existing = index_client.get_knowledge_base(definition.name)
    except ResourceNotFoundError:
        print("未作成:", definition.name)
        if not confirm_writes:
            raise RuntimeError("4.1 の計画を確認し、作成セルの CONFIRM_WRITES を True にして再実行してください。") from None
        return index_client.create_or_update_knowledge_base(definition, match_condition=MatchConditions.IfMissing)
    require_expected(existing.as_dict(), definition.as_dict())
    print("設定を照合して再利用:", definition.name)
    return existing

## 3. Entity Graph を直接読む

### 3.1 登録済みソースから実際のグラフ名とインデックス名を得る

管理されたグラフのインデックス名は固定値を推測せず、`createdResources.graphIndex` を読みます。`graphExtraction.graphName` はインデックス内のグラフを区別する名前です。同期状態も表示しますが、状態取得だけで抽出内容の正しさは判断しません。

この章の GQL は機能テスト 1 の内蔵オントロジーに合わせています。関係が未抽出の場合は、元の Notebook で同期・オントロジー・原文を確認してください。

出典: [Entity Graph の作成結果と同期確認](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#verify-extraction-completes)

In [ ]:
if RUN_AZURE:
    entity_source = index_client.get_knowledge_source(ENTITY_SOURCE).as_dict()
    if entity_source["kind"] != "azureBlob":
        raise ValueError("Entity Graph の azureBlob ソースを指定してください。")
    blob_parameters = entity_source["azureBlobParameters"]
    GRAPH_INDEX = blob_parameters["createdResources"]["graphIndex"]
    GRAPH_NAME = blob_parameters["graphExtraction"]["graphName"]
    if not GRAPH_INDEX or not GRAPH_NAME:
        raise ValueError("グラフ抽出設定と createdResources を確認してください。")
    table([{"ソース": ENTITY_SOURCE, "グラフ名": GRAPH_NAME, "グラフ インデックス": GRAPH_INDEX}])
    show_json(index_client.get_knowledge_source_status(ENTITY_SOURCE).as_dict())
else:
    print("未実行: 登録済み Entity Graph と同期状態の読み取り。")

### 3.2 グラフ定義からフィルターを作る

`graphProfile.kind = edgeList` を確認し、グラフ名フィールドも定義から取得します。OData フィルターを付けて、同じインデックスに別のグラフがある場合にも対象を限定します。

出典: [Documents - Graph の filter](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [公式 SDK のフィルター例](https://github.com/Azure/azure-sdk-for-python/blob/e1c0306c77b27b46c79755f53783474a03fd8c0d/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_AZURE:
    graph_schema = index_client.get_index(GRAPH_INDEX).as_dict()
    profile = graph_schema["graphProfile"]
    if profile["kind"] != "edgeList":
        raise ValueError("直接 GQL には edgeList の graphProfile が必要です。")
    graph_field = profile["fields"]["graphNameField"]
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", graph_field):
        raise ValueError("グラフ名フィールドの識別子を確認してください。")
    escaped_name = GRAPH_NAME.replace("'", "''")
    GRAPH_FILTER = f"{graph_field} eq '{escaped_name}'"
    graph_client = SearchClient(SEARCH_ENDPOINT, GRAPH_INDEX, credential, **client_options)
    azure_clients.append(graph_client)
    show_json(profile)
    print("対象フィルター:", GRAPH_FILTER)
else:
    print("未実行: グラフ定義とフィルターの確認。")

### 3.3 施設を経由する 2 ホップを GQL で探索する

「指定 ← 登録された施設 → 所在地」という経路を取得します。`registeredAs` と `locatedIn` はこのリポジトリの内蔵オントロジーの述語です。矢印の向きはオントロジーの関係方向に合わせています。施設のノードには型ラベルを付けず、2 つの述語の向きだけで限定します。

直接 GQL は検索モデルによる自然言語変換を行いません。まず保存された関係を確認することで、後半の自然言語検索が空だったときに「関係がない」のか「生成 GQL が合っていない」のかを分けて調べられます。

`LIMIT 10` は返すグラフ行の上限です。`maxEvidencePerRow` は各行の根拠収集の上限で、**別の値**です。直接 GQL の `hydrateDocuments` には `top` を指定しません。

検索後は、返された行を使って経路を SVG で描きます。描画のために Azure を再検索することはありません。

出典: [直接 GQL と根拠取得](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [REST Graph API](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
DIRECT_GQL = (
    "MATCH (designation:Designation)<-[:registeredAs]-(site)-[:locatedIn]->(place:Place) "
    "RETURN designation.name AS designation, site.name AS site, place.name AS place LIMIT 10"
)
print(DIRECT_GQL)
if RUN_AZURE:
    direct_result = graph_client.graph(
        gql=DIRECT_GQL, filter=GRAPH_FILTER, include_statistics=True, include_diagnostics=True,
        provenance=GraphQueryProvenanceParameters(
            kind="hydrate", max_evidence_per_row=6,
            hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(
                select="uid,snippet,snippet_parent_id,blob_url"),
        ),
    ).as_dict()
    table(direct_result["value"])
    show_json({"診断": direct_result.get("@search.graphDiagnostics"),
               "統計": direct_result.get("@search.graphStats")})
    if any(item.get("severity") == "error" for item in direct_result.get("@search.graphDiagnostics", [])):
        raise RuntimeError("グラフ診断にエラーがあります。上の詳細を確認してください。")
    if not direct_result["value"]:
        print("経路は 0 件です。抽出状態と型・述語を確認してください。抽出成功の証明にはなりません。")
else:
    print("未実行: Azure の直接 GQL。上に表示したのは要求する GQL だけです。")

In [ ]:
def draw_paths(paths: list[list[str]], predicates: list[str], directions: list[int]) -> None:
    if len(predicates) != len(directions) or any(direction not in (-1, 1) for direction in directions):
        raise ValueError("各関係の矢印の向きを 1 または -1 で指定してください。")
    if any(len(path) != len(predicates) + 1 for path in paths):
        raise ValueError("経路のノード数は関係数 + 1 である必要があります。")
    if not paths:
        print("描画できる経路は 0 件です。")
        return
    width, height = 300 * (len(predicates) + 1), 110 * len(paths)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="検索で見つかったグラフ経路">',
             '<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#345"/></marker></defs>']
    for row_number, path in enumerate(paths):
        y = row_number * 110 + 60
        for column, name in enumerate(path):
            x = column * 300 + 15
            parts.append(f'<rect x="{x}" y="{y - 24}" width="190" height="48" rx="8" fill="#e8f2ff" stroke="#3976b8"/>')
            parts.append(f'<text x="{x + 95}" y="{y + 5}" text-anchor="middle" font-family="sans-serif" font-size="14">{html.escape(str(name))}</text>')
        for column, predicate in enumerate(predicates):
            start, end = column * 300 + 207, (column + 1) * 300 + 12
            if directions[column] == -1:
                start, end = end, start
            parts.append(f'<line x1="{start}" y1="{y}" x2="{end}" y2="{y}" stroke="#345" stroke-width="2" marker-end="url(#arrow)"/>')
            parts.append(f'<text x="{column * 300 + 260}" y="{y - 10}" text-anchor="middle" font-family="sans-serif" font-size="12">{html.escape(predicate)}</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if RUN_AZURE:
    draw_paths([[row["designation"], row["site"], row["place"]] for row in direct_result["value"]],
               ["registeredAs", "locatedIn"], [-1, 1])
else:
    print("未実行: Azure が返した経路の描画。")

### 3.4 グラフ行から根拠チャンクへ戻る

各行の `@search.graphProvenance.evidence` にエッジ・チャンク・親文書の ID があります。`@search.graphProvenanceDocuments` は複数の行で共有される原文チャンクの辞書です。`chunkId` をキーに読みます。

表の原文は先頭 200 文字まで表示し、続きは「…」で省略します。全文は `evidence_rows` と `direct_result` に残ります。

`evidenceCount.relation = exact` は件数が確定していること、`lowerBound` は収集上限などにより下限しか分からないことを表します。根拠本文を取得できなかった場合は、その事実を表示します。

出典: [Entity Graph の根拠 ID](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [GraphQueryEvidenceCountRelation](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview#graphqueryevidencecountrelation)

In [ ]:
if RUN_AZURE:
    hydrated = direct_result.get("@search.graphProvenanceDocuments", {})
    evidence_rows = []
    for row_number, row in enumerate(direct_result["value"], start=1):
        provenance = row.get("@search.graphProvenance", {})
        print("行", row_number, "根拠件数:", provenance.get("evidenceCount", "未取得"))
        for evidence in provenance.get("evidence", []):
            chunk = hydrated.get(evidence["chunkId"], {})
            evidence_rows.append({"行": row_number, **evidence,
                                  "原文": chunk.get("snippet", "未取得"), "原文 URL": chunk.get("blob_url", "未取得")})
    table([{**row, "原文": preview_text(row["原文"])} for row in evidence_rows])
    if direct_result["value"] and not evidence_rows:
        print("検索行はありますが根拠 ID を取得できていません。provenance 設定と応答を確認してください。")
else:
    print("未実行: GQL の根拠と原文の確認。")

## 4. Entity Graph を Agentic Retrieval から使う

### 4.1 グラフ検索用ナレッジ ベースの計画

**現行のプレビューでは、Entity Graph 検索を既存のナレッジ ベースに後から有効化できません。** 初回は専用の `ft-entity-agentic-kb` を作ります。事前に表示される計画のソースとモデルを確認し、次の作成セルで `CONFIRM_WRITES = True` にしてください。同名のグラフ用 KB がすでにある場合は、計画と一致するときだけ再利用します。別名を使う場合は `.env` に `FOUNDRY_IQ_ENTITY_KB` を設定します。

グラフを抽出したチャットモデルとは別に、検索用モデルを設定します。同じデプロイを再利用しても、ナレッジ ベース側の設定は必要です。`AZURE_OPENAI_MODEL` には実モデル名を指定してください。デプロイ名と同じとは限りません。公式資料は `gpt-5.6-luna` と `low` を推奨していますが、ここでは利用者の実際の対応デプロイを使います。

`outputMode = extractiveData` として、まず検索データと出典を観察します。自然言語の最終回答をモデルで生成する工程は追加しません。

出典: [Entity Graph の制限](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#limitations) / [ナレッジ ベースの作成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base)

In [ ]:
if RUN_AZURE:
    entity_kb_plan = KnowledgeBase(
        name=ENTITY_KB, description="agentic-retrieval-graphs notebook: entity graph retrieval",
        knowledge_sources=[KnowledgeSourceReference(name=ENTITY_SOURCE)],
        models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=MODEL_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL))],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(), output_mode="extractiveData",
        retrieval_instructions="Use graph_search for entity relationship questions. Traverse the graph and return evidence for the relationships.",
    )
    show_json(entity_kb_plan.as_dict())
    if not MODEL_ENDPOINT or not CHAT_DEPLOYMENT or not CHAT_MODEL:
        print("モデル設定は未完了です。ライブ実行前に .env の実エンドポイント・デプロイ名・モデル名を設定してください。")
else:
    print("未実行: SDK モデルによる Entity Graph 用 KB の計画。")

In [ ]:
CONFIRM_WRITES = False

if RUN_AZURE:
    if not all((MODEL_ENDPOINT, CHAT_DEPLOYMENT, CHAT_MODEL)) or any("<" in value for value in (MODEL_ENDPOINT, CHAT_DEPLOYMENT, CHAT_MODEL)):
        raise ValueError("Azure OpenAI の実際のエンドポイント・チャット デプロイ・モデル名が必要です。")
    entity_kb_definition = create_or_reuse_kb(entity_kb_plan, confirm_writes=CONFIRM_WRITES)
    require_expected(entity_kb_definition.as_dict(), entity_kb_plan.as_dict())
    entity_kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=ENTITY_KB, **client_options)
    azure_clients.append(entity_kb_client)
    print("グラフ用ナレッジ ベースの設定を確認:", ENTITY_KB)
else:
    print("未実行: グラフ用ナレッジ ベースの作成・照合。")

### 4.2 自然言語の要求 JSON を作る

3 章と同じ施設・指定・所在地の関係を、今度は自然言語で質問します。`includeQuery = true` は生成 GQL を観察するため、`includeActivity = true` は実行履歴のための指定です。

ナレッジ ベースの `hydrateDocuments.top = 10` は取得する根拠文書数の上限で、GQL のグラフ行数の上限ではありません。`hydrate = False` に変えて要求を作り直すと原文取得を省略できますが、再検索には追加のモデル呼び出しが発生します。

出典: [グラフを使う retrieve 要求](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base) / [公式要求・応答例](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
ENTITY_QUESTION = "所在地ごとに、どの施設がどの指定や登録を受けていますか。施設名、指定名、所在地を、関係の根拠とともに示してください。"


def entity_request(question: str, hydrate: bool = True) -> KnowledgeBaseRetrievalRequest:
    if not question.strip():
        raise ValueError("質問を入力してください。")
    graph_parameters = AgentRuntimeGraphParams(include_query=True)
    if hydrate:
        graph_parameters.provenance = AgentRuntimeGraphProvenanceParams(
            kind="hydrate", hydrate_documents=AgentRuntimeGraphHydrateDocumentsParams(
                select="uid,snippet,snippet_parent_id,blob_url", top=10),
        )
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
        include_activity=True, output_mode="extractiveData", max_runtime_in_seconds=60,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        knowledge_source_params=[AzureBlobKnowledgeSourceParams(
            knowledge_source_name=ENTITY_SOURCE, include_references=True, include_reference_source_data=True,
            fail_on_error=True, graph_parameters=graph_parameters)],
    )


if RUN_AZURE:
    entity_payload = entity_request(ENTITY_QUESTION)
    show_json(entity_payload.as_dict())
else:
    print("未実行: SDK モデルによる Entity Graph の検索要求。")

In [ ]:
if RUN_AZURE:
    entity_result = entity_kb_client.retrieve(entity_payload, raw_response_hook=reject_partial_response).as_dict()
    print("応答のトップレベル項目:", list(entity_result))
    show_json(entity_result["response"])
else:
    print("未実行: Entity Graph の Agentic Retrieval。ライブの結果はまだありません。")

### 4.3 activity ログから処理内容と時間を可視化する

取得済みの `entity_result["activity"]` を、**全件の JSON → 処理別の一覧 → 時間の図**の順に表示します。可視化のための再検索やモデル呼び出しは行いません。`graphSearch` だけに絞らず、検索計画や推論の記録も確認します。

| ログの種類 | 確認する処理 |
|---|---|
| `modelQueryPlanning` | モデルによる検索計画と入力・出力トークン数 |
| `graphSearch` | 自然言語から GQL への変換、対象グラフ、試行数、成否、モデル処理時間 |
| `searchIndex` / `azureBlob` | ナレッジ ソースの検索と検索引数 |
| `agenticReasoning` | 取得時の推論設定と推論トークン数 |
| `modelAnswerSynthesis` | 回答生成。この Notebook の `extractiveData` では存在するとは限りません |

`startedAt`・`completedAt` があれば、その時刻でタイムラインを描き、並行していた処理も見えるようにします。開始・終了時刻がないログでは `elapsedMs` の比較図を表示し、実際の順番や並列性は推測しません。`queryTime` を開始時刻として代用することもしません。

図の時間範囲は記録された処理の範囲であり、API 応答全体の所要時間ではありません。処理時間の合計も API の所要時間とは限りません。トークン数は各ログの報告値を表示し、未記録を 0 に置き換えません。`graphSearchArguments.modelElapsedMs` はグラフ処理全体とは別の列に表示します。

3 章で手書きした GQL と、今回生成された GQL を比較してください。同じ質問でも、射影列や探索の組み立てが一致するとは限りません。以下は `graphSearch` がない場合や失敗を報告した場合に停止し、通常検索だけの応答をグラフ検索の成功として扱いません。

出典: [Activity array の種類と指標](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#activity-array) / [開始・終了時刻と処理の重なり](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#event-lifecycle) / [graphSearch の応答例](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
import math
from datetime import datetime


def display_activity(activities: list[dict], title: str, *, show_raw: bool = True) -> None:
    if not isinstance(activities, list):
        raise TypeError("activity は配列である必要があります。SDK の応答を確認してください。")
    print(title, "activity 全件:")
    if show_raw:
        show_json(activities)
    else:
        raw = html.escape(json.dumps(activities, ensure_ascii=False, indent=2))
        display(HTML(f'<details><summary>activity の元 JSON を開く</summary><pre>{raw}</pre></details>'))
    if not activities:
        print("activity は 0 件です。処理内容を確認できません。include_activity を確認してください。")
        return
    labels = {
        "modelQueryPlanning": "検索計画の生成", "graphSearch": "GQL 生成・グラフ検索",
        "searchIndex": "インデックス検索", "azureBlob": "Blob ソース検索",
        "agenticReasoning": "推論設定・使用量", "modelAnswerSynthesis": "回答生成",
        "modelWebSummarization": "Web 検索結果の要約", "imageServing": "画像の取得・受け渡し",
    }

    def timestamp(value: str | None) -> datetime | None:
        if value is None:
            return None
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        if parsed.tzinfo is None:
            raise ValueError("activity の時刻にタイムゾーンがありません。")
        return parsed

    rows, records = [], []
    for activity in activities:
        kind = activity.get("type") or "未記録"
        graph = activity.get("graphSearchArguments") or {}
        model = activity.get("model") or {}
        start, end = timestamp(activity.get("startedAt")), timestamp(activity.get("completedAt"))
        if start is not None and end is not None and end < start:
            raise ValueError(f"activity {activity.get('id')}: 終了時刻が開始時刻より前です。")
        elapsed = activity.get("elapsedMs")
        basis = "elapsedMs"
        if elapsed is None and start is not None and end is not None:
            elapsed = (end - start).total_seconds() * 1000
            basis = "開始・終了時刻から算出"
        if elapsed is not None and (isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0):
            raise ValueError(f"activity {activity.get('id')}: elapsedMs が不正です。")

        def metric(key: str) -> object:
            value = activity.get(key)
            return graph.get(key) if value is None else value

        row = {
            "ID": activity.get("id"), "種類": kind, "処理": labels.get(kind, "未登録の処理種類"),
            "ソース": activity.get("knowledgeSourceName"), "モデル": model.get("modelName") or graph.get("modelName"),
            "時間 ms": elapsed, "時間の根拠": basis if elapsed is not None else None,
            "モデル時間 ms": graph.get("modelElapsedMs"),
            "入力トークン": metric("inputTokens"), "出力トークン": metric("outputTokens"),
            "推論トークン": activity.get("reasoningTokens"), "件数": activity.get("count"),
            "試行数": graph.get("attempts"), "成否・エラー": activity.get("error") or graph.get("outcome"),
        }
        rows.append({key: "未記録" if value is None else value for key, value in row.items()})
        records.append((activity, start, end, elapsed))
    table(rows)
    bounds = [(start, end) for _, start, end, _ in records if start is not None and end is not None]
    origin = min(start for start, _ in bounds) if bounds else None
    span = (max(end for _, end in bounds) - origin).total_seconds() * 1000 if origin is not None else max((elapsed for _, _, _, elapsed in records if elapsed is not None), default=0)
    caption = "タイムライン: 最初の開始時刻からの経過 ms" if origin is not None else "時間比較: 実際の開始時刻・順番・並列性は未記録"
    print(caption)
    left, plot_width, height = 330, 550, 95 + 44 * len(records)
    scale = plot_width / max(span, 1)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1040 {height}" width="100%" role="img" aria-label="{html.escape(title)}の処理時間">',
             f'<text x="10" y="22" font-family="sans-serif" font-size="14">{html.escape(caption)}</text>']
    ticks = [span * index / 4 for index in range(5)] if span else [0]
    for tick in ticks:
        x = left + tick * scale
        parts.append(f'<line x1="{x}" y1="45" x2="{x}" y2="{height - 25}" stroke="#ddd"/>')
        parts.append(f'<text x="{x}" y="40" text-anchor="middle" font-family="sans-serif" font-size="12">{tick:.1f}</text>')
    for index, (activity, start, end, elapsed) in enumerate(records):
        y = 70 + 44 * index
        kind = activity.get("type") or "未記録"
        label = f"{activity.get('id', '未記録')}: {labels.get(kind, kind)}"
        parts.append(f'<text x="10" y="{y + 5}" font-family="sans-serif" font-size="13">{html.escape(label)}</text>')
        if origin is not None and start is not None and end is not None:
            offset, duration = (start - origin).total_seconds() * 1000, (end - start).total_seconds() * 1000
        elif origin is None and elapsed is not None:
            offset, duration = 0, elapsed
        else:
            parts.append(f'<text x="{left}" y="{y + 5}" fill="#64748b" font-family="sans-serif" font-size="12">開始・終了時刻未記録</text>')
            continue
        x, width = left + offset * scale, duration * scale
        color = "#dc2626" if activity.get("error") or ((activity.get("graphSearchArguments") or {}).get("outcome") not in (None, "Success")) else "#3976b8"
        if duration == 0:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{color}"/>')
        else:
            parts.append(f'<rect x="{x}" y="{y - 9}" width="{width}" height="18" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{left + plot_width + 12}" y="{y + 5}" font-family="sans-serif" font-size="12">{duration:.1f} ms</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if RUN_AZURE:
    display_activity(entity_result["activity"], "Entity Graph の Agentic Retrieval")
    graph_activities = [activity for activity in entity_result["activity"] if activity["type"] == "graphSearch"]
    if not graph_activities:
        show_json(entity_result["activity"])
        raise RuntimeError("graphSearch の実行を確認できません。新規 KB、ソース設定、検索指示を確認してください。")
    for activity in graph_activities:
        arguments = activity["graphSearchArguments"]
        print("activity ID:", activity["id"], "生成 GQL:", arguments.get("gql", "未取得"))
        if arguments.get("outcome") != "Success":
            raise RuntimeError("グラフ処理の成功を確認できません。上の実行履歴を確認してください。")
else:
    print("未実行: graphSearch の実行履歴と生成 GQL の確認。")

### 4.4 参照に入っているグラフ行を読む

グラフ行は `references[].sourceData.rows` に入る場合があります。**`response` が空でも、公式例では `references` に結果が入っています。** `response` の有無だけで、検索が失敗したと判断しないでください。

生成 GQL の射影列はモデルに依存するので、手書き GQL の `designation`・`site`・`place` 列があるとは決めつけず、実際の列を表示します。`activitySource` を使って、どの検索から得た参照かを追跡します。

出典: [グラフ行と参照の応答例](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
if RUN_AZURE:
    retrieved_graph_rows = []
    activity_by_id = {activity["id"]: activity for activity in entity_result["activity"]}
    for reference in entity_result["references"]:
        source_data = reference.get("sourceData")
        if reference["type"] == "graphSearch" and isinstance(source_data, dict) and "rows" in source_data:
            source_activity = activity_by_id[reference["activitySource"]]
            table([{"参照 ID": reference["id"], "activity ID": source_activity["id"],
                    "グラフ": reference["graphIndexName"], "列": source_data.get("columns", "未取得")}])
            table(source_data["rows"])
            retrieved_graph_rows.extend(source_data["rows"])
    print("参照から読み取ったグラフ行数:", len(retrieved_graph_rows))
    if not retrieved_graph_rows:
        print("グラフ行を確認できません。実行履歴と references の生データを確認してください。")
        show_json(entity_result["references"])
else:
    print("未実行: 参照に含まれるグラフ行の確認。")

### 4.5 根拠チャンクと元文書の URL を読む

原文を含む参照も `type = graphSearch` なので、型だけではグラフ行と区別できません。`sourceData.snippet` のある参照を取り出します。`uid` はチャンク、`snippet_parent_id` は親文書、`blob_url` は元文書の場所です。

表の原文は先頭 200 文字まで表示し、続きは「…」で省略します。全文は `chunk_references` と `entity_result` に残ります。

同じ `activitySource` は同じ検索処理に由来することを示しますが、原文参照が特定のグラフ行を一対一で証明するとは限りません。行ごとのエッジ対応を詳しく調べる場合は、3.4 の直接 GQL の根拠を使います。Blob URL は公開 URL とは限らず、閲覧には権限が必要な場合があります。

出典: [ナレッジ ベースのグラフ参照と原文取得](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base)

In [ ]:
if RUN_AZURE:
    chunk_references = []
    for reference in entity_result["references"]:
        source_data = reference.get("sourceData")
        if reference["type"] == "graphSearch" and isinstance(source_data, dict) and "snippet" in source_data:
            chunk_references.append({"参照 ID": reference["id"], "activity ID": reference["activitySource"],
                                     "チャンク ID": source_data.get("uid", "未取得"),
                                     "親文書 ID": source_data.get("snippet_parent_id", "未取得"),
                                     "原文 URL": source_data.get("blob_url", "未取得"), "原文": source_data["snippet"]})
    table([{**row, "原文": preview_text(row["原文"])} for row in chunk_references])
    if not chunk_references:
        print("原文チャンクを取得できていません。hydrate 設定とソースのフィールドを確認してください。")
else:
    print("未実行: Agentic Retrieval が返した根拠原文の確認。")

## 5. 結果の読み方と切り分け

| 観察した状態 | 次に確認すること |
|---|---|
| `graphSearch` がない | Entity Graph は新規 KB が必要です。グラフ対応ソースを作成時に接続したか、検索指示、実際の activity を確認します |
| 直接 GQL でも 0 行 | 同期完了、抽出された型・述語、グラフ名、元記事を確認します。空の結果から抽出成功は判断できません |
| 直接 GQL は行を返すが自然言語検索は空 | 生成 GQL と手書き GQL を比較し、型名・述語・名称条件の違いを確認します |
| `502` と `All retrieval tasks failed` | 検索モデルの `429` が原因のことがあります。モデルの利用枠とサービスのエラーを確認し、無制限に再送しません |
| `403` | 実行ユーザーの Search 権限と、Search のマネージド ID のモデル権限、ネットワーク条件を分けて確認します |
| `206` | 部分応答です。この Notebook は成功として受け入れません。表示された本文と要求 ID から調べます |
| `412` | 同名の KB が別の操作で作成された可能性があります。4.1 の計画と既存 KB の定義を照合し、必要なら別名を指定します。無条件の書き込みに変更しません |
| 原文がない | `includeReferenceSourceData`、`hydrate`、ソースの原文フィールドを確認します。直接 GQL と KB の hydration の設定差にも注意します |

### 保存される変更と終了後の扱い

Azure に保存される変更は、4.1 で初回作成を許可した場合の専用ナレッジ ベースだけです。既存の Blob ソースやグラフには変更を加えません。下のセルはクライアントを閉じるだけで、KB はカーネル終了後も残ります。

削除は自動化していません。不要になった場合は、この Notebook で作成した KB を参照するアプリやエージェントがないことを確認し、その KB だけを削除してください。再利用したリソースは削除しません。

グラフや根拠チャンクを取得できる範囲が、元文書の閲覧権限と一致するとは限りません。利用を許可されたソースだけを対象にし、元文書とグラフの両方のアクセス権を確認してください。Notebook の実行結果には原文や接続先が含まれ得るため、共有やコミットの前に出力をクリアしてください。

出典: [Entity Graph の障害切り分け](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#troubleshoot-graph-queries) / [検索時の権限](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [同時実行制御](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE:
    for client in azure_clients:
        client.close()
    credential.close()
    print("SDK と資格情報のクライアントを閉じました。Azure リソースは削除していません。")
else:
    print("Azure の資格情報クライアントは作成していません。")